# FinViz Financial News Sentiment Analysis
**Automated News Scraping, Sentiment Analysis & Trading Recommendations**

This notebook scrapes financial news from FinViz, performs sentiment analysis, and generates buy/sell recommendations.

## Features
- Real-time news scraping from FinViz
- Sentiment analysis with TextBlob
- Trading recommendations: STRONG BUY, BUY, HOLD, SELL, STRONG SELL
- Confidence scores
- Interactive visualizations
- Portfolio analysis

**Disclaimer:** This tool is for educational purposes only. Do not use as sole basis for trading decisions.

In [1]:
# Install required packages (run once)
# !pip install requests beautifulsoup4 pandas textblob matplotlib seaborn plotly -q
# !python -m textblob.download_corpora

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime
from textblob import TextBlob
import time
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('All libraries imported successfully!')

In [3]:
class FinVizSentimentAnalyzer:
    def __init__(self):
        self.base_url = 'https://finviz.com/quote.ashx?t='
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    def scrape_news(self, ticker: str, max_articles: int = 20) -> List[Dict]:
        url = f'{self.base_url}{ticker}'
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            news_table = soup.find('table', id='news-table')
            if not news_table:
                print(f'No news found for {ticker}')
                return []
            news_data = []
            rows = news_table.findAll('tr')
            current_date = None
            for row in rows[:max_articles]:
                td_date = row.find('td', align='right')
                if td_date:
                    date_text = td_date.text.strip()
                    if len(date_text.split()) > 1:
                        current_date = date_text.split()[0]
                        time_str = date_text.split()[1]
                    else:
                        time_str = date_text
                td_news = row.find('td', align='left')
                if td_news:
                    link = td_news.find('a')
                    if link:
                        headline = link.text.strip()
                        source = td_news.find('span')
                        source_text = source.text.strip() if source else 'Unknown'
                        news_data.append({'date': current_date, 'time': time_str, 'headline': headline, 'source': source_text})
            print(f'Scraped {len(news_data)} articles for {ticker}')
            return news_data
        except Exception as e:
            print(f'Error: {e}')
            return []

    def analyze_sentiment(self, text: str) -> Tuple[float, float]:
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity

    def calculate_sentiment_score(self, news_data: List[Dict]) -> Dict:
        if not news_data:
            return {'avg_polarity': 0, 'avg_subjectivity': 0, 'positive_count': 0, 'negative_count': 0, 'neutral_count': 0, 'total_articles': 0, 'polarities': []}
        polarities = []
        subjectivities = []
        positive_count = negative_count = neutral_count = 0
        for news in news_data:
            polarity, subjectivity = self.analyze_sentiment(news['headline'])
            polarities.append(polarity)
            subjectivities.append(subjectivity)
            news['polarity'] = polarity
            news['subjectivity'] = subjectivity
            if polarity > 0.1:
                positive_count += 1
                news['sentiment'] = 'Positive'
            elif polarity < -0.1:
                negative_count += 1
                news['sentiment'] = 'Negative'
            else:
                neutral_count += 1
                news['sentiment'] = 'Neutral'
        return {
            'avg_polarity': np.mean(polarities),
            'avg_subjectivity': np.mean(subjectivities),
            'positive_count': positive_count,
            'negative_count': negative_count,
            'neutral_count': neutral_count,
            'total_articles': len(news_data),
            'polarities': polarities
        }

    def generate_recommendation(self, sentiment_metrics: Dict) -> Dict:
        avg_polarity = sentiment_metrics['avg_polarity']
        positive_ratio = sentiment_metrics['positive_count'] / max(sentiment_metrics['total_articles'], 1)
        if avg_polarity > 0.15 and positive_ratio > 0.5:
            action = 'STRONG BUY'
            confidence = min(abs(avg_polarity) * 100, 95)
        elif avg_polarity > 0.05:
            action = 'BUY'
            confidence = min(abs(avg_polarity) * 80, 75)
        elif avg_polarity < -0.15 and positive_ratio < 0.3:
            action = 'STRONG SELL'
            confidence = min(abs(avg_polarity) * 100, 95)
        elif avg_polarity < -0.05:
            action = 'SELL'
            confidence = min(abs(avg_polarity) * 80, 75)
        else:
            action = 'HOLD'
            confidence = 50
        return {'action': action, 'confidence': round(confidence, 2), 'sentiment_score': round(avg_polarity, 4)}

    def analyze_ticker(self, ticker: str, max_articles: int = 20) -> Dict:
        print('='*60)
        print(f' Analyzing {ticker.upper()} ')
        print('='*60)
        news_data = self.scrape_news(ticker, max_articles)
        if not news_data:
            return {'ticker': ticker, 'error': 'No news data found'}
        sentiment_metrics = self.calculate_sentiment_score(news_data)
        recommendation = self.generate_recommendation(sentiment_metrics)
        return {
            'ticker': ticker.upper(),
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'news_data': news_data,
            'sentiment_metrics': sentiment_metrics,
            'recommendation': recommendation
        }

analyzer = FinVizSentimentAnalyzer()
print('FinVizSentimentAnalyzer initialized!')

In [5]:
# QUICK START: Analyze Single Stock
# Replace 'AAPL' with any ticker symbol

ticker = 'AAPL'
analysis = analyzer.analyze_ticker(ticker=ticker, max_articles=20)

# Display summary
if 'error' not in analysis:
    metrics = analysis['sentiment_metrics']
    rec = analysis['recommendation']
    print('='*60)
    print(f' Analysis Time: {analysis["timestamp"]}')
    print(f' Articles Analyzed: {metrics["total_articles"]}')
    print(f' Average Score: {metrics["avg_polarity"]:.4f}')
    print(f' Positive: {metrics["positive_count"]} ({metrics["positive_count"]/metrics["total_articles"]*100:.1f}%)') 
    print(f' Negative: {metrics["negative_count"]} ({metrics["negative_count"]/metrics["total_articles"]*100:.1f}%)') 
    print(f' RECOMMENDATION: {rec["action"]}')
    print(f' Confidence: {rec["confidence"]}%')
    print('='*60)